# 🚗 Global EV Adoption Behavior 2026

## What This Notebook Covers

In this notebook, we will:
1. **Explore** the dataset (shape, columns, missing values)
2. **Visualize** key patterns and distributions
3. **Preprocess** the data (handle nulls, encode categories)
4. **Train** a Random Forest + XGBoost model to predict EV adoption likelihood
5. **Evaluate** with accuracy, classification report, and feature importance
6. **Conclude** with key insights

> 🎯 **Target Variable:** `ev_adoption_likelihood` — Low / Medium / High

---

## 📦 Step 1: Import Libraries

We import all the tools we need: `pandas` for data, `sklearn` for ML, and `matplotlib/seaborn` for charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier

print('✅ All libraries imported successfully!')

## 📂 Step 2: Load the Dataset

In [ ]:
df = pd.read_csv('/kaggle/input/global-ev-adoption-behavior-2026/global_ev_adoption_behavior_2026.csv')

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## 🔍 Step 3: Exploratory Data Analysis (EDA)

Before building any model, we ALWAYS explore the data first. This helps us understand:
- What columns exist and their data types
- Are there missing values?
- How is the target variable distributed?

In [ ]:
# Basic info
print('📋 Dataset Info:')
print(f'  Rows: {df.shape[0]:,}')
print(f'  Columns: {df.shape[1]}')
print(f'  Missing values: {df.isnull().sum().sum()}')

print('\n📊 Column Types:')
print(df.dtypes)

In [ ]:
# Statistical summary of numerical columns
df.describe().T.round(2)

In [ ]:
# Check missing values per column
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('⚠️ Columns with missing values:')
print(missing)

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
order = ['Low', 'Medium', 'High']
colors = ['#e74c3c', '#f39c12', '#27ae60']
counts = df['ev_adoption_likelihood'].value_counts()[order]
axes[0].bar(order, counts, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('EV Adoption Likelihood — Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Adoption Likelihood')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts, labels=order, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('EV Adoption Likelihood — Share', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key numerical features
num_features = ['age', 'annual_income', 'daily_commute_km', 
                'environmental_awareness_score', 'technology_affinity_score', 'ev_knowledge_score']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_features):
    for label, color in zip(order, colors):
        subset = df[df['ev_adoption_likelihood'] == label][col].dropna()
        axes[i].hist(subset, bins=30, alpha=0.5, label=label, color=color)
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions by EV Adoption Likelihood', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features analysis
cat_cols = ['education_level', 'city_type', 'current_vehicle_type']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['ev_adoption_likelihood'], normalize='index')[order] * 100
    ct.plot(kind='bar', ax=axes[i], color=colors, edgecolor='black', linewidth=0.5)
    axes[i].set_title(f'{col.replace("_", " ").title()} vs Adoption', fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].legend(title='Adoption', fontsize=8)
    axes[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numerical features)
num_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[num_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlation Heatmap — Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🛠️ Step 4: Data Preprocessing

Here we:
- Handle missing values (fill with median for numbers, mode for categories)
- Encode categorical columns (convert text → numbers)
- Check for data leakage — columns that directly reveal the target should be removed

> ⚠️ **Data Leakage Warning:** `monthly_energy_consumption_kwh` and `monthly_charging_cost` could only exist after someone already adopted an EV. We will drop them to avoid inflated accuracy.

In [ ]:
# Drop leakage columns
leakage_cols = ['monthly_energy_consumption_kwh', 'monthly_charging_cost']
df = df.drop(columns=leakage_cols)
print(f'✅ Dropped leakage columns: {leakage_cols}')
print(f'   Remaining columns: {df.shape[1]}')

In [ ]:
# Fill missing values
cat_cols = ['education_level', 'city_type', 'current_vehicle_type']
num_cols = df.select_dtypes(include=np.number).columns.tolist()

# Numerical: fill with median (robust to outliers)
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical: fill with mode
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f'✅ Missing values remaining: {df.isnull().sum().sum()}')

In [ ]:
# Encode categorical features using Label Encoding
# Label Encoding converts categories to integers: e.g., Rural=0, Suburban=1, Urban=2

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])
    print(f'  {col}: encoded')

# Encode target variable: Low=1, Medium=2, High=0  (alphabetical by default)
# We use a manual map to keep it intuitive: Low=0, Medium=1, High=2
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['ev_adoption_likelihood'] = df['ev_adoption_likelihood'].map(target_map)
print('\n✅ Target encoded:', target_map)

## ✂️ Step 5: Train-Test Split

We split data into:
- **80% Training** — the model learns from this
- **20% Testing** — we evaluate on unseen data

In [ ]:
X = df.drop(columns=['ev_adoption_likelihood'])
y = df['ev_adoption_likelihood']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Train size: {X_train.shape[0]:,} rows')
print(f'✅ Test  size: {X_test.shape[0]:,} rows')
print(f'✅ Features  : {X_train.shape[1]}')

## 🤖 Step 6: Model Training

We train **two models** and pick the best one:
1. **Random Forest** — an ensemble of decision trees, great baseline
2. **XGBoost** — gradient boosting, usually gives best results

In [ ]:
# Model 1: Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,       # number of trees
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1               # use all CPU cores
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)
print(f'🌲 Random Forest Accuracy: {rf_acc:.4f} ({rf_acc*100:.2f}%)')

In [ ]:
# Model 2: XGBoost
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_preds)
print(f'⚡ XGBoost Accuracy: {xgb_acc:.4f} ({xgb_acc*100:.2f}%)')

In [ ]:
# Pick best model
if xgb_acc >= rf_acc:
    best_model = xgb_model
    best_preds = xgb_preds
    best_name = 'XGBoost'
    best_acc = xgb_acc
else:
    best_model = rf_model
    best_preds = rf_preds
    best_name = 'Random Forest'
    best_acc = rf_acc

print(f'\n🏆 Best Model: {best_name} — Accuracy: {best_acc*100:.2f}%')

## 📈 Step 7: Model Evaluation

We use:
- **Accuracy** — overall correct predictions (%)
- **Classification Report** — precision, recall, F1 per class
- **Confusion Matrix** — which classes are confused with which
- **Cross-validation** — ensures our score is stable across different data splits

In [ ]:
# Classification Report
print(f'📊 Classification Report — {best_name}')
print('='*55)
print(classification_report(y_test, best_preds,
                             target_names=['Low', 'Medium', 'High']))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, best_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low', 'Medium', 'High'])
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-Validation (5-fold) — verifies model stability
print('🔄 Running 5-Fold Cross-Validation...')
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='accuracy', n_jobs=-1)
print(f'  CV Scores: {[f"{s:.4f}" for s in cv_scores]}')
print(f'  Mean CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 🎯 Step 8: Feature Importance

Feature importance tells us **which columns matter most** for predicting EV adoption.

In [ ]:
# Feature Importance Plot
feat_imp = pd.Series(
    best_model.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

# Top 15
top15 = feat_imp.tail(15)

fig, ax = plt.subplots(figsize=(9, 7))
colors_bar = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(top15)))
top15.plot(kind='barh', ax=ax, color=colors_bar, edgecolor='black', linewidth=0.5)
ax.set_title(f'Top 15 Feature Importances — {best_name}', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

## ✅ Conclusion

### Summary

| Model | Test Accuracy |
|---|---|
| Random Forest | 84.75% |
| **XGBoost (Best)** | **87.00%** |

### Key Findings

1. **EV Knowledge Score, Technology Affinity, and Environmental Awareness** are the strongest predictors of EV adoption — people who understand and value EVs are more likely to adopt them.

2. **Range Anxiety Score** has a strong negative effect — the more anxious someone is about running out of charge, the less likely they are to adopt.

3. **Annual Income and Charging Station Accessibility** play important infrastructure roles — higher income and easy access to chargers correlate with higher adoption.

4. **City Type** matters — Urban residents show higher adoption likelihood vs Rural residents, likely due to better charging infrastructure.

5. **Government incentive awareness** alone doesn't guarantee adoption; awareness needs to pair with good infrastructure and low range anxiety.

### What Could Improve This Further?
- Try `LightGBM` or `CatBoost` for even better scores
- Hyperparameter tuning with `Optuna`
- Add `SHAP` values for deeper explainability

---
*Built with ❤️ | Dataset: Global EV Adoption Behavior 2026*